# Aprendizaje Estadístico y Data Mining

## Práctica 3: Especies de monos

### Objetivo
El etiquetado de imágenes es una tarea ardua. Es por ello y también debido a sus aplicaciones prácticas que los científicos llevan un tiempo intentando mejorar los métodos para clasificarlas automáticamente. En la aduana del aeropuerto de Madrid se intenta luchar contra el tráfico de animales exóticos. Para ello se va a crear un clasificador que realizando una foto a un animal (en este caso monos) pueda decidir si pertenece a una especie en peligro de extinción o no. Dicho clasificador funcionará mediante un set de entrenamiento donde se buscará un plano que divida las diferentes clases dispuesta en un espacio n-dimensional dependiendo de sus características. Muestra todos los resultados del algoritmo paso a paso.

Para ello usaremos el dataset **“monos.zip”** que se encuentra en *scikit-learn*. Elige el clasificador que más se adapte de entre los vistos en clase y usa *scikit-learn* junto con las librerías que necesites para resolver las siguientes cuestiones.

* [Link al dataset.](https://www.kaggle.com/datasets/slothkong/10-monkey-species?select=validation)

**Enunciado:**  Crea un clasificador que permita saber qué especie de mono es a partir de una imagen. Realiza al menos dos configuraciones y dibuja una tabla donde se muestre la precisión con la que clasifican.

**Solución**



In [ ]:
# Importamos las librerías necesarias
import os
import numpy as np
import pandas as pd
from PIL import Image
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Definimos la ruta del archivo CSV
ENTRENAMIENTO_PATH = "data/monos_especies/training"
VALIDACION_PATH = "data/monos_especies/validation"

def cargar_datos(ruta_base: str, tamano_imagen=(64, 64)) -> tuple:
    """
    Carga los datos de una carpeta y los convierte en un array de numpy.
    Cada imagen se convierte a un array y se normaliza (0-1).

    Args:
        ruta_base (str): Ruta base donde se encuentran las carpetas de imágenes.
        tamano_imagen (tuple, optional): Tamaño al que se redimensionarán las imágenes. Por defecto (64, 64).

    Returns:
        tuple: Un tuple que contiene dos arrays de numpy:
            * datos: Array de imágenes vectorizadas.
            * etiquetas: Array de etiquetas correspondientes a las imágenes.
    """

    datos = []
    etiquetas = []

    # 1. Recorremos las carpetas de la ruta base
    for carpeta in os.listdir(ruta_base):
        etiqueta = carpeta
        carpeta_path = os.path.join(ruta_base, carpeta)

        # 2. Para cada carpeta, recorremos los archivos de imagen
        for fichero in os.listdir(carpeta_path):

            # 3. Generamos la ruta completa de la imagen
            imagen_path = os.path.join(carpeta_path, fichero)
            try:
                # 4. Abrimos la imagen para convertirla a un array
                imagen = Image.open(imagen_path).convert('RGB')

                # 5. Redimensionamos la imagen para que tenga el tamaño especificado
                imagen = imagen.resize(tamano_imagen)

                # 6. Convertimos la imagen a un array de numpy, lo normalizamos y lo vectorizamos
                # para que cada imagen tenga un tamaño fijo
                array_imagen = np.asarray(imagen) / 255.0  # Normalizamos
                datos.append(array_imagen.flatten())       # Vectorizamos

                # 7. Añadimos la etiqueta correspondiente a la lista de etiquetas
                # (la etiqueta es el nombre de la carpeta)
                etiquetas.append(etiqueta)
            except:
                pass # Ignoramos errores al abrir imágenes

    # 8. Devolvemos los datos y etiquetas como arrays de numpy
    return np.array(datos), np.array(etiquetas)


X_entrenamiento, y_entrenamiento = cargar_datos(ENTRENAMIENTO_PATH)
X_validacion, y_validacion = cargar_datos(VALIDACION_PATH)

# Codificamos las etiquetas de texto a números
# para que los clasificadores puedan trabajar con ellas
labelEncoder = LabelEncoder()
y_entrenamiento_codificado = labelEncoder.fit_transform(y_entrenamiento)
y_validacion_codificado = labelEncoder.transform(y_validacion)

# 1. Support Vector Machine (SVM) con kernel lineal

# 1.1 Inicializamos el clasificador SVM con un kernel lineal
support_vector_machine_lineal = SVC(kernel='linear')
# 1.2 Entrenamos el clasificador SVM con los datos de entrenamiento
support_vector_machine_lineal.fit(X_entrenamiento, y_entrenamiento_codificado)
# 1.3 Realizamos predicciones sobre los datos de validación
y_predicha_svm = support_vector_machine_lineal.predict(X_validacion)
# 1.4 Calculamos la precisión del clasificador SVM comparando las etiquetas predichas con las etiquetas reales
accuracy_svm = accuracy_score(y_validacion_codificado, y_predicha_svm)


# 2. Support Vector Machine (SVM) con kernel no lineal

# 2.1 Inicializamos el clasificador SVM con un kernel no lineal
support_vector_machine_no_lineal = SVC(kernel='rbf')
# 2.2 Entrenamos el clasificador SVM con los datos de entrenamiento
support_vector_machine_no_lineal.fit(X_entrenamiento, y_entrenamiento_codificado)
# 2.3 Realizamos predicciones sobre los datos de validación
y_predicha_svm_no_lineal = support_vector_machine_no_lineal.predict(X_validacion)
# 2.4 Calculamos la precisión del clasificador SVM comparando las etiquetas predichas con las etiquetas reales
accuracy_svm_no_lineal = accuracy_score(y_validacion_codificado, y_predicha_svm_no_lineal)

# 3.1 Inicializamos el clasificador de regresión logística
# con un número máximo de iteraciones de 1000
regresion = LogisticRegression(max_iter=1000)
# 3.2 Entrenamos el clasificador de regresión logística con los datos de entrenamiento
regresion.fit(X_entrenamiento, y_entrenamiento_codificado)
# 3.3 Realizamos predicciones sobre los datos de validación
y_predicha_regresion = regresion.predict(X_validacion)
# 3.4 Calculamos la precisión del clasificador de regresión logística
accuracy_regresion = accuracy_score(y_validacion_codificado, y_predicha_regresion)

# 4. K-Nearest Neighbors (KNN)
# 4.1 Inicializamos el clasificador con k=5
knn = KNeighborsClassifier(n_neighbors=5)
# 4.2 Entrenamos el clasificador con los datos de entrenamiento
knn.fit(X_entrenamiento, y_entrenamiento_codificado)
# 4.3 Realizamos predicciones sobre los datos de validación
y_predicha_knn = knn.predict(X_validacion)
# 4.4 Calculamos la precisión del clasificador KNN
accuracy_knn = accuracy_score(y_validacion_codificado, y_predicha_knn)

# Tabla de resultados con la precisión de cada clasificador
df_resultados = pd.DataFrame({
    "Clasificador": ["SVM (lineal)", "SVM (no lineal)", "Regresión logística", "KNN (k=5)"],
    "Precisión (validación)": [
        f"{accuracy_svm:.2%}",
        f"{accuracy_svm_no_lineal:.2%}",
        f"{accuracy_regresion:.2%}",
        f"{accuracy_knn:.2%}"
    ]
})

display(df_resultados)

**Enunciado:** Elige 5 imágenes de diferentes especies que no hayas usado ni para entrenar el modelo, ni para evaluarlo y clasifícalas. Usa para ello el modelo que mejor clasifique de los del punto anterior. Índica con que error ha funcionado el clasificador.

**Solución**

In [ ]:
MONOS_FAMOSOS_PATH = "data/monos_famosos"

X_prueba_monos_famosos, especies_estimadas, rutas_imagenes = cargar_datos(MONOS_FAMOSOS_PATH)

# Elegimos el mejor modelo de los que hemos entrenado
mejor_modelo_nombre = df_resultados.sort_values("Precisión (validación)", ascending=False).iloc[0]["Clasificador"]

if mejor_modelo_nombre == "SVM (lineal)":
    mejor_modelo = support_vector_machine_lineal
elif mejor_modelo_nombre == "SVM (no lineal)":
    mejor_modelo = support_vector_machine_no_lineal
elif mejor_modelo_nombre == "Regresión logística":
    mejor_modelo = regresion
elif mejor_modelo_nombre == "KNN (k=5)":
    mejor_modelo = knn
else:
    raise ValueError("Clasificador no reconocido.")

# Predecimos las especies de los 10 monos famosos
y_predicha_monos_famosos = mejor_modelo.predict(X_prueba_monos_famosos)
etiquetas_predichas = labelEncoder.inverse_transform(y_predicha_monos_famosos)

# Visualizamos los resultados
plt.figure(figsize=(15, 5))
for mono_famoso in range(len(X_prueba_monos_famosos)):
    plt.subplot(1, 5, mono_famoso+1)
    imagen = Image.open(rutas_imagenes[mono_famoso])
    plt.imshow(imagen)
    plt.axis('off')
    plt.title(f"Predicción:\n{etiquetas_predichas[mono_famoso]}")
plt.suptitle("Clasificación de monos famosos")
plt.tight_layout()
plt.show()

# Imprimimos los resultados
print("Resultados:")
for mono_famoso in range(len(X_prueba_monos_famosos)):
    print(f"Imagen {mono_famoso+1}: archivo = {os.path.basename(rutas_imagenes[mono_famoso])} → Predicha: {etiquetas_predichas[mono_famoso]}")
